<a href="https://colab.research.google.com/github/23MH1A05L3/AI-workshop/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai pydantic

In [2]:
import os
import getpass

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass("Gemini API key: ")

Gemini API key: ··········


In [3]:
from pydantic import BaseModel
from typing import List, Optional

In [19]:
class Education(BaseModel):
    degree: str
    institution: str
    year: int

In [20]:
class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float


In [6]:
from google import genai
from pydantic import ValidationError


In [10]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
def extract_resume(raw_text: str, max_retries: int = 1):
    pass

In [11]:
import os
print(os.listdir())

['.config', 'Resume.txt', '.ipynb_checkpoints', 'sample_data']


In [12]:
with open('Resume.txt', 'r', encoding='utf-8') as f:
    print(f.read()[:2000])

Rama Pithani
Phone:+91-7036136496 | Email:ramapithani154@gmail.com | LinkedIn:Rama Pithani | GitHub:RamaPithani
Summary
Computer Science undergraduate with strong problem-solving abilities and a solid foundation in C/C++, Data
Structures, Algorithms and SQL. Seeking an opportunity to apply technical knowledge, enhance software
development skills, and contribute effectively while gaining industry experience.
Projects
JobLens– Centralized Campus Drive Management System | GitHub Link | React.js, Node.js, Express, MongoDB
[Feb 2026]
• Led end-to-end requirements gathering by analyzing campus placement workflows and identifying 24 functional
requirements across Coordinator and Student roles.
• Designed the complete system architecture and MongoDB schema with 8 collections, optimized
relationships/indexes, and resolved the MongoDB parallel-array index constraint.
• Defined the MERN + JWT + Multer + Nodemailer tech stack based on scalability and feature requirements
including file upload, asy

In [13]:
try:
    bad = extract_resume('')
    print('Unexpected success')
except Exception as e:
    print('Caught gracefully:', type(e).__name__)
    print('Message:', str(e)[:200])

Unexpected success


In [15]:
resume_files = [
    'Resume.txt',
    'Resume1.txt',
    'Resume2.txt'
]

resumes = []

for file in resume_files:
    with open(file, 'r', encoding='utf-8') as f:
        resumes.append(f.read())

print(f"Loaded {len(resumes)} resumes")

Loaded 3 resumes


In [16]:
results = []

for i, r in enumerate(resumes):
    try:
        parsed = extract_resume(r)

        results.append(parsed)

        print(
            f"\nResume {i+1}: "
            f"{parsed.name} — "
            f"{len(parsed.skills)} skills, "
            f"{parsed.experience_years} years exp"
        )

    except Exception as e:
        print(
            f"\nResume {i+1}: FAILED — "
            f"{type(e).__name__}: {str(e)[:200]}"
        )


Resume 1: FAILED — AttributeError: 'NoneType' object has no attribute 'name'

Resume 2: FAILED — AttributeError: 'NoneType' object has no attribute 'name'

Resume 3: FAILED — AttributeError: 'NoneType' object has no attribute 'name'


In [17]:
with open('Resume.txt', 'r', encoding='utf-8') as f:
    text = f.read()

result = extract_resume(text)

print(result)
print(type(result))

None
<class 'NoneType'>


In [21]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1):

    if not raw_text.strip():
        raise ValueError("Resume text is empty")

    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=f'''
Extract a Resume JSON from this text.
Return ONLY JSON.

{raw_text}
''',
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )

            return Resume.model_validate_json(resp.text)

        except ValidationError as e:

            if attempt == max_retries:
                raise

            fix_prompt = (
                f"Fix this JSON to match schema. "
                f"Errors: {e}. "
                f"Original: {resp.text}"
            )

            resp = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=fix_prompt,
                config={
                    'response_mime_type': 'application/json',
                    'response_schema': Resume.model_json_schema(),
                },
            )

            return Resume.model_validate_json(resp.text)

In [22]:
with open('Resume.txt', 'r', encoding='utf-8') as f:
    text = f.read()

result = extract_resume(text)

print(result)
print(type(result))

name='Rama Pithani' email='ramapithani154@gmail.com' phone='+91-7036136496' education=[Education(degree='B.Tech in Computer Science and Engineering', institution='Aditya College of Engineering and Technology, Surampalem', year=2027), Education(degree='Class XII', institution='Sri Chaitanya College, Kakinada', year=2023)] skills=['C++', 'C', 'HTML', 'CSS', 'Java Script', 'React', 'Node.js', 'Express', 'Postman', 'SQL', 'MongoDB', 'Github', 'Docker', 'Data Structures and Algorithms', 'OOPS', 'Operating Systems', 'Database Management Systems'] projects=['JobLens– Centralized Campus Drive Management System', 'Library Management System API'] experience_years=0.0
<class '__main__.Resume'>


In [23]:
resume_files = [
    'Resume.txt',
    'Resume1.txt',
    'Resume2.txt'
]

resumes = []

for file in resume_files:
    with open(file, 'r', encoding='utf-8') as f:
        resumes.append(f.read())

results = []

for i, r in enumerate(resumes):
    parsed = extract_resume(r)

    print(
        f"Resume {i+1}: "
        f"{parsed.name} — "
        f"{len(parsed.skills)} skills, "
        f"{parsed.experience_years} years exp"
    )

Resume 1: Rama Pithani — 17 skills, 0.0 years exp
Resume 2: Candidate Name — 10 skills, 0.0 years exp
Resume 3:  — 14 skills, 0.0 years exp


In [26]:
results = []

for i, r in enumerate(resumes):
    parsed = extract_resume(r)
    results.append(parsed)

    print(
        f"Resume {i+1}: "
        f"{parsed.name} — "
        f"{len(parsed.skills)} skills, "
        f"{parsed.experience_years} years exp"
    )

Resume 1: Rama Pithani — 17 skills, 0.0 years exp
Resume 2: Unknown — 10 skills, 0.0 years exp
Resume 3: Unknown Name — 16 skills, 0.0 years exp


In [27]:
print(len(results))

3


In [28]:
print(results[0].model_dump_json(indent=2))

{
  "name": "Rama Pithani",
  "email": "ramapithani154@gmail.com",
  "phone": "+91-7036136496",
  "education": [
    {
      "degree": "B.Tech in Computer Science and Engineering",
      "institution": "Aditya College of Engineering and Technology, Surampalem",
      "year": 2027
    },
    {
      "degree": "Class XII",
      "institution": "Sri Chaitanya College, Kakinada",
      "year": 2023
    }
  ],
  "skills": [
    "C++",
    "C",
    "HTML",
    "CSS",
    "Java Script",
    "React",
    "Node.js",
    "Express",
    "Postman",
    "SQL",
    "MongoDB",
    "Github",
    "Docker",
    "Data Structures and Algorithms",
    "OOPS",
    "Operating Systems",
    "Database Management Systems"
  ],
  "projects": [
    "JobLens– Centralized Campus Drive Management System",
    "Library Management System API"
  ],
  "experience_years": 0.0
}


In [29]:
print(results[1].model_dump_json(indent=2))

{
  "name": "Unknown",
  "email": "Unknown",
  "phone": null,
  "education": [
    {
      "degree": "B.Tech in Computer Science and Engineering",
      "institution": "Aditya College of Engineering and Technology, Surampalem",
      "year": 2023
    },
    {
      "degree": "Class XII",
      "institution": "Sri Chaitanya College, Kakinada",
      "year": 2023
    }
  ],
  "skills": [
    "C++",
    "C",
    "Java",
    "Python",
    "HTML",
    "CSS",
    "SQL",
    "Data Structures and Algorithms",
    "Leadership",
    "TeamWork"
  ],
  "projects": [
    "Tic-Tac-Toe Game with AI",
    "Huffman File Compression Tool"
  ],
  "experience_years": 0.0
}


In [30]:
print(results[2].model_dump_json(indent=2))

{
  "name": "Unknown Name",
  "email": "unknown@example.com",
  "phone": null,
  "education": [
    {
      "degree": "B.Tech in Computer Science and Engineering",
      "institution": "Aditya College of Engineering and Technology, Surampalem",
      "year": 2027
    },
    {
      "degree": "Class XII",
      "institution": "Sri Chaitanya College, Kakinada",
      "year": 2023
    }
  ],
  "skills": [
    "C++",
    "C",
    "Java",
    "HTML",
    "CSS",
    "JavaScript",
    "Basic React",
    "Node.js",
    "Express",
    "SQL",
    "MongoDB",
    "Data Structures and Algorithms",
    "Operating Systems",
    "Database Management Systems",
    "Sequelize",
    "MySQL"
  ],
  "projects": [
    "Library Management System API - Developed a RESTful API for managing books, members, transactions, and fines with state machines, business rules, and a normalized relational schema.",
    "Budget Buddy- Finance App - A collaborative web project to track income/expenses, set savings goals, and